In [4]:
input_file = r"../data/gene\TableS2-FinalData_BENZALKONIUM-1 - UNSPECIFIED_genes.xlsx"


In [5]:
import pandas as pd

df = pd.read_excel(input_file)
print(df.head())


           Gene  BENZALKONIUM-1 - UNSPECIFIED
0  ECK2456-EUTP                     -1.593395
1  ECK2737-NLPD                      1.540203
2  ECK2623-YFJK                     -1.603921
3  ECK2272-NUOL                     -1.981765
4  ECK0664-NAGC                     -2.409324


In [12]:
import pandas as pd
import cobra
import difflib
import os
import re


# ===== File paths =====
excel_file = r"../data/gene\TableS2-FinalData_BENZALKONIUM-1 - UNSPECIFIED_genes.xlsx"
model_file = r"../models/iJO1366.xml"
output_file = r"../data/mapped-gene\mapped_gene_scores.xlsx"

# ===== Read Excel file =====
df = pd.read_excel(excel_file)
print("Excel 列名：", list(df.columns))

# Assume one column is named "Gene" and another "Score"
# If your column names differ, edit here
gene_col = "Gene"
score_col = df.columns[1]

gene_scores = df[[gene_col, score_col]].dropna()
print(f"读取 {len(gene_scores)} 个基因分数")

# ===== Load the iJO1366 model =====
print("载入 iJO1366 模型中...")
model = cobra.io.read_sbml_model(model_file)
print(f"模型载入成功，包含 {len(model.genes)} 个基因")

# ===== Build model gene mapping table =====
gene_map = {}
for g in model.genes:
    gene_map[g.id.lower()] = g.id
    if hasattr(g, 'name') and g.name:
        gene_map[g.name.lower()] = g.id

# ===== Match genes =====
def normalize_name(g):
    return re.sub(r"[^a-z0-9]", "", g.lower())

mapped = []
unmapped = []
for gene, score in gene_scores.itertuples(index=False):
    g_lower = normalize_name(str(gene))
    match = re.match(r"eck(\d+)([a-z0-9]*)", g_lower)
    possible_ids = []
    if match:
        num, suffix = match.groups()
        possible_ids = [f"b{num}", suffix, f"eck{num}"]
    else:
        possible_ids = [g_lower]

    found = False
    for pid in possible_ids:
        pid_norm = normalize_name(pid)
        if pid_norm in gene_map:
            mapped.append((gene, gene_map[pid_norm], score))
            found = True
            break
        else:
            m = difflib.get_close_matches(pid_norm, gene_map.keys(), n=1, cutoff=0.7)
            if m:
                mapped.append((gene, gene_map[m[0]], score))
                found = True
                break

    if not found:
        unmapped.append(gene)
      

# ===== Write output =====
mapped_df = pd.DataFrame(mapped, columns=["YourGene", "GEM_Gene_ID", "Score"])
mapped_df.to_excel(output_file, index=False)

print(f"\n✅ 匹配完成：{len(mapped_df)} 个成功匹配")
print(f"❌ 未匹配：{len(unmapped)} 个")
if unmapped:
    print("示例未匹配基因：", unmapped[:10])

# Optional: save the unmatched list
unmapped_file = os.path.splitext(output_file)[0] + "_unmapped.xlsx"
pd.DataFrame(unmapped, columns=["UnmatchedGene"]).to_excel(unmapped_file, index=False)
print(f"\n已保存映射结果到：\n{output_file}")
print(f"未匹配基因列表到：\n{unmapped_file}")


Excel 列名： ['Gene', 'BENZALKONIUM-1 - UNSPECIFIED']
读取 595 个基因分数
载入 iJO1366 模型中...
模型载入成功，包含 1367 个基因

✅ 匹配完成：594 个成功匹配
❌ 未匹配：1 个
示例未匹配基因： ['ECK1205/1207/1209-RDLABC']

已保存映射结果到：
../data/mapped-gene\mapped_gene_scores.xlsx
未匹配基因列表到：
../data/mapped-gene\mapped_gene_scores_unmapped.xlsx


In [13]:
import pandas as pd
import cobra
import difflib
import os
import re

# ===== Path setup =====
input_dir = r"../data/gene"
output_dir = r"../data/mapped-gene"
model_file = r"../models/iJO1366.xml"

os.makedirs(output_dir, exist_ok=True)

# ===== Load the iJO1366 model (once only) =====
print("载入 iJO1366 模型中...")
model = cobra.io.read_sbml_model(model_file)
print(f"模型载入成功，包含 {len(model.genes)} 个基因")

# ===== Build model gene mapping table =====
gene_map = {}
for g in model.genes:
    gene_map[g.id.lower()] = g.id
    if hasattr(g, 'name') and g.name:
        gene_map[g.name.lower()] = g.id

# ===== Name normalization function =====
def normalize_name(g):
    return re.sub(r"[^a-z0-9]", "", g.lower())

# ===== Single-file processing function =====
def map_genes_in_file(excel_path, output_dir):
    try:
        df = pd.read_excel(excel_path)
    except Exception as e:
        print(f"❌ 无法读取文件 {excel_path}: {e}")
        return

    if len(df.columns) < 2:
        print(f"⚠️ 文件 {excel_path} 没有足够的列")
        return

    gene_col = df.columns[0]
    score_col = df.columns[1]
    gene_scores = df[[gene_col, score_col]].dropna()
    print(f"\n处理文件：{os.path.basename(excel_path)}，读取 {len(gene_scores)} 个基因分数")

    mapped = []
    unmapped = []

    for gene, score in gene_scores.itertuples(index=False):
        g_lower = normalize_name(str(gene))
        match = re.match(r"eck(\d+)([a-z0-9]*)", g_lower)
        possible_ids = []
        if match:
            num, suffix = match.groups()
            possible_ids = [f"b{num}", suffix, f"eck{num}"]
        else:
            possible_ids = [g_lower]

        found = False
        for pid in possible_ids:
            pid_norm = normalize_name(pid)
            if pid_norm in gene_map:
                mapped.append((gene, gene_map[pid_norm], score))
                found = True
                break
            else:
                m = difflib.get_close_matches(pid_norm, gene_map.keys(), n=1, cutoff=0.7)
                if m:
                    mapped.append((gene, gene_map[m[0]], score))
                    found = True
                    break

        if not found:
            unmapped.append(gene)

    # ===== Output =====
    base_name = os.path.splitext(os.path.basename(excel_path))[0]
    mapped_file = os.path.join(output_dir, f"{base_name}_mapped.xlsx")
    unmapped_file = os.path.join(output_dir, f"{base_name}_unmapped.xlsx")

    mapped_df = pd.DataFrame(mapped, columns=["YourGene", "GEM_Gene_ID", "Score"])
    mapped_df.to_excel(mapped_file, index=False)

    pd.DataFrame(unmapped, columns=["UnmatchedGene"]).to_excel(unmapped_file, index=False)

    print(f"✅ 匹配完成：{len(mapped_df)} 个成功匹配，❌ 未匹配：{len(unmapped)} 个")
    return mapped_file, unmapped_file


# ===== Iterate over the whole folder =====
excel_files = [f for f in os.listdir(input_dir) if f.lower().endswith(('.xlsx', '.xls'))]

print(f"\n发现 {len(excel_files)} 个 Excel 文件，开始批量处理...\n")

for f in excel_files:
    excel_path = os.path.join(input_dir, f)
    map_genes_in_file(excel_path, output_dir)

print("\n🎉 所有文件处理完成！")


载入 iJO1366 模型中...
模型载入成功，包含 1367 个基因

发现 324 个 Excel 文件，开始批量处理...


处理文件：TableS2-FinalData_16C - _genes.xlsx，读取 546 个基因分数
✅ 匹配完成：546 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_18C - _genes.xlsx，读取 591 个基因分数
✅ 匹配完成：590 个成功匹配，❌ 未匹配：1 个

处理文件：TableS2-FinalData_20C - _genes.xlsx，读取 602 个基因分数
✅ 匹配完成：602 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_40C - _genes.xlsx，读取 616 个基因分数
✅ 匹配完成：616 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_42C - _genes.xlsx，读取 559 个基因分数
✅ 匹配完成：559 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_43.5C - _genes.xlsx，读取 561 个基因分数
✅ 匹配完成：561 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_45C - _genes.xlsx，读取 562 个基因分数
✅ 匹配完成：562 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_A22-0.5 - _genes.xlsx，读取 609 个基因分数
✅ 匹配完成：609 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_A22-15.0 - _genes.xlsx，读取 637 个基因分数
✅ 匹配完成：637 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_A22-2.0 - _genes.xlsx，读取 635 个基因分数
✅ 匹配完成：635 个成功匹配，❌ 未匹配：0 个

处理文件：TableS2-FinalData_A22-5.0 - _genes.xlsx，读取 648 个基因分数
✅ 匹配完成：648 个成功匹配，❌ 未匹配：0 个

处理文件：TableS